## Extraction (Api Rest) / constructor finance

In [ ]:

import time
from pathlib import Path
from typing import Dict, Any

import pandas as pd
import requests

SEASONS = [2021, 2022, 2023, 2024, 2025]
ERGAST = "https://api.jolpi.ca/ergast/f1"

PROJECT_DIR = Path(r"C:\Users\imane\Desktop\Fil-rouge")
DATA_DIR = PROJECT_DIR / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)

COST_CAP_BASELINE_USD = {
    2021: 145_000_000,
    2022: 140_000_000,
    2023: 135_000_000,
    2024: 135_000_000,
    2025: 135_000_000,
}

def get_json(url: str) -> Dict[str, Any]:
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

def get_constructor_standings(season: int) -> pd.DataFrame:
    url = f"{ERGAST}/{season}/constructorStandings.json"
    data = get_json(url)
    lists = data["MRData"]["StandingsTable"]["StandingsLists"]
    if not lists:
        return pd.DataFrame()

    standings = lists[0]["ConstructorStandings"]
    rows = []
    for s in standings:
        c = s["Constructor"]
        rows.append({
            "season": season,
            "constructorId": c["constructorId"],
            "constructorName": c["name"],
            "constructorRank": int(s["position"]),
            "constructorPoints": float(s["points"]),
        })
    return pd.DataFrame(rows)

def main():
    all_df = []
    for season in SEASONS:
        df = get_constructor_standings(season)
        if df.empty:
            continue
        df["costCapUSD"] = COST_CAP_BASELINE_USD.get(season)
        all_df.append(df)
        print(f"✔ Constructors {season}")
        time.sleep(0.2)

    final = pd.concat(all_df, ignore_index=True)
    out_path = DATA_DIR / "f1_constructor_finance_2021_2025.csv"
    final.to_csv(out_path, index=False, encoding="utf-8")
    print(f"✅ Saved: {out_path} ({len(final)} rows)")

if __name__ == "__main__":
    main()

## Extraction (web scraping) / Driver finance (Toutes les saisons)

In [ ]:



import re
import time
import unicodedata
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ================= CONFIG =================
SEASONS = [2021, 2022, 2023, 2024, 2025]
ERGAST = "https://api.jolpi.ca/ergast/f1"

PROJECT_DIR = Path(r"C:\Users\imane\Desktop\Fil-rouge")
DATA_DIR = PROJECT_DIR / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)

SALARY_SOURCES = {
    2021: "https://racingnews365.com/formula-1-driver-salaries-2021",
    2022: "https://racingnews365.com/what-are-the-driver-salaries-for-2022",
    2023: "https://racingnews365.com/f1-driver-salaries-2023",
    2024: "https://www.nbcnewyork.com/news/sports/how-much-are-f1-drivers-paid-salaries-2024/5192874/",
    2025: "https://racingnews365.com/2025-f1-driver-salaries-how-much-do-f1-drivers-earn",
}

# Corrections ponctuelles si besoin
NAME_OVERRIDES = {
    2025: {"bortoletto": "bortoleto"},  # exemple
}

# ================= HTTP =================
def get_json(url: str, retries: int = 6, backoff: float = 1.5) -> Dict[str, Any]:
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; F1PerformanceArchitect/1.0)",
        "Accept": "application/json",
    }
    last_err: Optional[Exception] = None
    for i in range(retries):
        try:
            r = requests.get(url, timeout=30, headers=headers)
            if r.status_code in (429, 403, 408, 500, 502, 503, 504):
                time.sleep(backoff * (i + 1))
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(backoff * (i + 1))
    raise last_err if last_err else RuntimeError(f"Échec requête: {url}")

def get_html(url: str) -> str:
    r = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    return r.text

# ================= NORMALISATION =================
def normalize_text(s: str) -> str:
    s = str(s).strip()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ================= ERGAST DRIVERS =================
def get_driver_standings(season: int) -> pd.DataFrame:
    url = f"{ERGAST}/{season}/driverStandings.json"
    data = get_json(url)
    lists = data["MRData"]["StandingsTable"]["StandingsLists"]
    if not lists:
        return pd.DataFrame()

    standings = lists[0]["DriverStandings"]
    rows = []
    for s in standings:
        d = s["Driver"]
        constructors = s.get("Constructors", [])
        constructor = constructors[0] if constructors else {}
        rows.append({
            "season": season,
            "driverId": d["driverId"],
            "givenName": d["givenName"],
            "familyName": d["familyName"],
            "driverName": f'{d["givenName"]} {d["familyName"]}',
            "constructorId": constructor.get("constructorId", ""),
            "driverRank": int(s["position"]),
            "driverPoints": float(s["points"]),
        })
    return pd.DataFrame(rows)

# ================= SALARY PARSING =================
def parse_salary_to_usd(raw: str) -> Tuple[Optional[float], Optional[float], Optional[float]]:
    """
    Retourne (min_million, max_million, mid_million) en MILLIONS.
    Ex: "$55 million" -> (55,55,55)
        "30m" -> (30,30,30)
        "10 - 12" -> (10,12,11)
    """
    if raw is None:
        return (None, None, None)
    s = str(raw).strip().lower()
    if s == "" or s in {"nan", "none"}:
        return (None, None, None)

    # nettoyer
    s = s.replace("$", "").replace("usd", "").replace("million", "").replace("millions", "")
    s = s.replace("m", "").replace(",", ".")
    s = re.sub(r"\s+", " ", s).strip()

    # range "10 - 12"
    m = re.match(r"^\s*(\d+(\.\d+)?)\s*-\s*(\d+(\.\d+)?)\s*$", s)
    if m:
        a = float(m.group(1))
        b = float(m.group(3))
        return (a, b, (a + b) / 2.0)

    # single number
    m = re.search(r"(\d+(\.\d+)?)", s)
    if m:
        a = float(m.group(1))
        return (a, a, a)

    return (None, None, None)

def pick_best_salary_table(soup: BeautifulSoup) -> Optional[Any]:
    tables = soup.find_all("table")
    if not tables:
        return None

    def score_table(t) -> int:
        th_text = " ".join(normalize_text(th.get_text(" ")) for th in t.find_all("th"))
        score = 0
        if "driver" in th_text or "name" in th_text:
            score += 3
        if "salary" in th_text or "earn" in th_text:
            score += 3
        if "team" in th_text:
            score += 1
        return score

    best = max(tables, key=score_table)
    if score_table(best) == 0:
        return None
    return best

def extract_salary_rows(season: int, url: str) -> pd.DataFrame:
    html = get_html(url)
    soup = BeautifulSoup(html, "html.parser")

    table = pick_best_salary_table(soup)
    if table is None:
        return pd.DataFrame(columns=["season", "driver_raw", "salary_raw", "source_url"])

    # headers
    headers = [normalize_text(th.get_text(" ", strip=True)) for th in table.find_all("th")]
    if not headers:
        return pd.DataFrame(columns=["season", "driver_raw", "salary_raw", "source_url"])

    # trouver indices colonnes
    driver_idx = None
    salary_idx = None
    for i, h in enumerate(headers):
        if driver_idx is None and ("driver" in h or "name" in h):
            driver_idx = i
        if salary_idx is None and ("salary" in h or "earn" in h):
            salary_idx = i

    if driver_idx is None or salary_idx is None:
        return pd.DataFrame(columns=["season", "driver_raw", "salary_raw", "source_url"])

    rows = []
    for tr in table.find_all("tr"):
        cells = [td.get_text(" ", strip=True) for td in tr.find_all(["td", "th"])]
        if len(cells) <= max(driver_idx, salary_idx):
            continue
        drv = cells[driver_idx]
        sal = cells[salary_idx]
        # ignorer la ligne header
        if normalize_text(drv) in {"driver", "name"}:
            continue
        if drv.strip() == "":
            continue
        rows.append({
            "season": season,
            "driver_raw": drv,
            "salary_raw": sal,
            "source_url": url
        })

    return pd.DataFrame(rows)

def map_salary_to_driverid(season: int, drivers_df: pd.DataFrame, salary_df: pd.DataFrame) -> pd.DataFrame:
    d = drivers_df.copy()
    d["given_norm"] = d["givenName"].map(normalize_text)
    d["family_norm"] = d["familyName"].map(normalize_text)

    family_to_ids = d.groupby("family_norm")["driverId"].apply(list).to_dict()

    def name_to_driverid(name_raw: str) -> str:
        if not isinstance(name_raw, str):
            return ""
        n = normalize_text(name_raw)

        # overrides
        for wrong, correct in NAME_OVERRIDES.get(season, {}).items():
            n = re.sub(rf"\b{wrong}\b", correct, n)

        tokens = n.split()
        if not tokens:
            return ""

        # souvent la page donne juste "Verstappen" -> on prend dernier token
        family = tokens[-1]
        ids = family_to_ids.get(family, [])
        if len(ids) == 1:
            return ids[0]

        # si on a prénom + nom
        if len(tokens) >= 2:
            given = tokens[0]
            cand = d[(d["family_norm"] == family) & (d["given_norm"] == given)]
            if len(cand) == 1:
                return cand.iloc[0]["driverId"]

        return ""

    out = salary_df.copy()
    out["driverId"] = out["driver_raw"].apply(name_to_driverid)

    mins, maxs, mids = [], [], []
    for raw in out["salary_raw"].tolist():
        mn, mx, md = parse_salary_to_usd(raw)
        mins.append(mn); maxs.append(mx); mids.append(md)

    out["salary_million_min"] = mins
    out["salary_million_max"] = maxs
    out["salary_million_mid"] = mids
    out["estimatedSalaryUSD"] = out["salary_million_mid"].apply(
        lambda x: int(round(x * 1_000_000)) if pd.notna(x) and x is not None else pd.NA
    )

    out = out[out["driverId"] != ""].copy()
    return out[["season", "driverId", "salary_raw", "estimatedSalaryUSD", "source_url"]]

# ================= MAIN BUILD =================
def build_driver_finance_2021_2025() -> Path:
    all_years = []

    for season in SEASONS:
        drivers = get_driver_standings(season)
        if drivers.empty:
            print(f"⚠️ No drivers standings for {season}")
            continue

        url = SALARY_SOURCES[season]
        salary_rows = extract_salary_rows(season, url)

        if salary_rows.empty:
            print(f"⚠️ No salary table extracted for {season} ({url})")
            drivers["salary_raw"] = pd.NA
            drivers["estimatedSalaryUSD"] = pd.NA
            drivers["salarySourceURL"] = pd.NA
        else:
            mapped = map_salary_to_driverid(season, drivers, salary_rows)

            drivers = drivers.merge(
                mapped[["season", "driverId", "salary_raw", "estimatedSalaryUSD", "source_url"]],
                on=["season", "driverId"],
                how="left"
            )
            drivers.rename(columns={"source_url": "salarySourceURL"}, inplace=True)

        final_cols = [
            "season", "driverId", "driverName", "constructorId",
            "driverRank", "driverPoints",
            "salary_raw", "estimatedSalaryUSD", "salarySourceURL"
        ]
        all_years.append(drivers[final_cols].copy())
        print(f"✅ {season}: drivers={len(drivers)} | salaries_found={drivers['estimatedSalaryUSD'].notna().sum()}")
        time.sleep(0.5)

    df_final = pd.concat(all_years, ignore_index=True)
    out_path = DATA_DIR / "f1_driver_finance_2021_2025.csv"
    df_final.to_csv(out_path, index=False, encoding="utf-8")
    print(f"\n🎯 Saved: {out_path} ({len(df_final)} rows)")
    return out_path

if __name__ == "__main__":
    build_driver_finance_2021_2025()



✅ 2021: drivers=21 | salaries_found=20
✅ 2022: drivers=22 | salaries_found=20
✅ 2023: drivers=22 | salaries_found=19
⚠️ No salary table extracted for 2024 (https://www.nbcnewyork.com/news/sports/how-much-are-f1-drivers-paid-salaries-2024/5192874/)
✅ 2024: drivers=24 | salaries_found=0
✅ 2025: drivers=21 | salaries_found=20

🎯 Saved: C:\Users\imane\Desktop\Fil-rouge\data\raw\f1_driver_finance_2021_2025.csv (110 rows)


C:\Users\imane\AppData\Local\Temp\ipykernel_4264\3956217260.py:490: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(all_years, ignore_index=True)


### Extraction saison 2024

In [ ]:
import re
import unicodedata
import pandas as pd
import requests

NBC_2024 = "https://www.nbcnewyork.com/news/sports/how-much-are-f1-drivers-paid-salaries-2024/5192874/"
IN_PATH  = r"C:\Users\imane\Desktop\Fil-rouge\data\raw\f1_driver_finance_2021_2025.csv"
OUT_PATH = r"C:\Users\imane\Desktop\Fil-rouge\data\raw\f1_driver_finance_2021_2025.csv"  # overwrite ok

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

def normalize_text(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", s.lower())

def parse_salary_to_usd(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    s = str(x).strip().lower().replace(",", "")
    m = re.search(r"(\d+(\.\d+)?)", s)
    if not m:
        return None
    val = float(m.group(1))
    # NBC affiche "$55 million" => millions
    if "million" in s or s.endswith("m") or re.search(r"\bm\b", s):
        return val * 1_000_000
    # si déjà en dollars
    if val > 1_000_000:
        return val
    # fallback: souvent les sites donnent en millions
    return val * 1_000_000

def fetch_tables(url: str) -> list[pd.DataFrame]:
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return pd.read_html(r.text)

def extract_nbc_2024_salary() -> pd.DataFrame:
    # 1) URL normale
    tables = []
    try:
        tables = fetch_tables(NBC_2024)
    except Exception:
        pass

    # 2) fallback AMP (souvent marche)
    if not tables:
        amp = NBC_2024.rstrip("/") + "/amp/"
        tables = fetch_tables(amp)

    if not tables:
        raise RuntimeError("Aucune table HTML trouvée sur NBC (même en /amp/)")

    # --- FIX: NBC peut donner des colonnes [0,1,2,3] (pas d'en-tête <th>) ---
    def fix_header(df: pd.DataFrame) -> pd.DataFrame:
        df2 = df.copy()

        # Si colonnes numériques, essayer de promouvoir la 1ère ligne en header
        if all(isinstance(c, int) for c in df2.columns):
            first_row = df2.iloc[0].astype(str).str.strip().str.lower().tolist()
            # Heuristique: la première ligne ressemble à un header
            if any(("name" in x) or ("driver" in x) for x in first_row) and any("salary" in x for x in first_row):
                df2.columns = df2.iloc[0].astype(str).str.strip()
                df2 = df2.iloc[1:].reset_index(drop=True)

        # Nettoyage colonnes
        df2.columns = [str(c).strip() for c in df2.columns]
        return df2

    fixed_tables = [fix_header(t) for t in tables]

    # choisir la table la plus probable (contient Name + Salary)
    def score(df: pd.DataFrame) -> int:
        cols = [str(c).lower() for c in df.columns]
        s = 0
        if any("name" in c or "driver" in c for c in cols): s += 3
        if any("salary" in c for c in cols): s += 3
        if any("team" in c for c in cols): s += 1
        if len(df) >= 15: s += 1
        return s

    best = sorted(fixed_tables, key=score, reverse=True)[0].copy()

    # détecter colonnes
    name_col = next((c for c in best.columns if "name" in str(c).lower() or "driver" in str(c).lower()), None)
    sal_col  = next((c for c in best.columns if "salary" in str(c).lower()), None)

    if name_col is None or sal_col is None:
        # debug minimal mais utile
        raise RuntimeError(
            f"Colonnes inattendues dans la table NBC après fix_header: {list(best.columns)}"
        )

    out = pd.DataFrame({
        "season": 2024,
        "driverName": best[name_col].astype(str).str.strip(),
        "salary_raw": best[sal_col].astype(str).str.strip(),
    })
    out["driverNorm"] = out["driverName"].apply(normalize_text)
    out["estimatedSalaryUSD"] = out["salary_raw"].apply(parse_salary_to_usd)
    out["salarySourceURL"] = NBC_2024
    return out
   

# ---------- MAIN PATCH ----------
df = pd.read_csv(IN_PATH)

# garder uniquement season=2024
df_2024 = df[df["season"] == 2024].copy()
if df_2024.empty:
    raise RuntimeError("Ton CSV ne contient aucune ligne season=2024")

sal_2024 = extract_nbc_2024_salary()

df_2024["driverNorm"] = df_2024["driverName"].apply(normalize_text)

patched = df_2024.merge(
    sal_2024[["driverNorm", "salary_raw", "estimatedSalaryUSD", "salarySourceURL"]],
    on="driverNorm",
    how="left",
    suffixes=("", "_new")
)

# on remplit uniquement si on a trouvé une valeur
for col in ["salary_raw", "estimatedSalaryUSD", "salarySourceURL"]:
    patched[col] = patched[col + "_new"].combine_first(patched[col])
    patched.drop(columns=[col + "_new"], inplace=True)

patched.drop(columns=["driverNorm"], inplace=True)

# réinjecter dans le dataset complet
df_out = pd.concat([df[df["season"] != 2024], patched], ignore_index=True)

df_out.to_csv(OUT_PATH, index=False)
print("✅ Patch 2024 terminé.")
print("Salaries non-null en 2024:", df_out[df_out["season"] == 2024]["estimatedSalaryUSD"].notna().sum())

### Test après extraction

In [ ]:

import pandas as pd
from pathlib import Path

df = pd.read_csv(r"C:\Users\imane\Desktop\Fil-rouge\data\raw\f1_driver_finance_2021_2025.csv")

print("Rows:", df.shape[0])
print("Salary filled:", df["estimatedSalaryUSD"].notna().sum())
print("By season:")
print(df.groupby("season")["estimatedSalaryUSD"].apply(lambda s: s.notna().sum()))
df.sort_values(["season","driverRank"]).head(10)

Rows: 110
Salary filled: 98
By season:
season
2021    20
2022    20
2023    19
2024    19
2025    20
Name: estimatedSalaryUSD, dtype: int64


,season,driverId,driverName,constructorId,driverRank,driverPoints,salary_raw,estimatedSalaryUSD,salarySourceURL
0,2021,max_verstappen,Max Verstappen,red_bull,1,395.5,$25m,25000000.0,https://racingnews365.com/formula-1-driver-sal...
1,2021,hamilton,Lewis Hamilton,mercedes,2,387.5,$30m,30000000.0,https://racingnews365.com/formula-1-driver-sal...
2,2021,bottas,Valtteri Bottas,mercedes,3,226.0,$10m,10000000.0,https://racingnews365.com/formula-1-driver-sal...
3,2021,perez,Sergio Pérez,red_bull,4,190.0,$8m,8000000.0,https://racingnews365.com/formula-1-driver-sal...
4,2021,sainz,Carlos Sainz,ferrari,5,164.5,$10m,10000000.0,https://racingnews365.com/formula-1-driver-sal...
5,2021,norris,Lando Norris,mclaren,6,160.0,$5m,5000000.0,https://racingnews365.com/formula-1-driver-sal...
6,2021,leclerc,Charles Leclerc,ferrari,7,159.0,$12m,12000000.0,https://racingnews365.com/formula-1-driver-sal...
7,2021,ricciardo,Daniel Ricciardo,mclaren,8,115.0,$15m,15000000.0,https://racingnews365.com/formula-1-driver-sal...
8,2021,gasly,Pierre Gasly,alphatauri,9,110.0,$5m,5000000.0,https://racingnews365.com/formula-1-driver-sal...
9,2021,alonso,Fernando Alonso,alpine,10,81.0,$20m,20000000.0,https://racingnews365.com/formula-1-driver-sal...
